# Step 1: Data Exploration & Inspection ("Learn the Data")

This notebook scans the `data/` directory, checks sample counts, class distributions (Real vs Fake), sampling rates, audio durations, channel configurations, and logs unreadable or corrupted audio files.

In [1]:
import os
import json
import logging
from pathlib import Path
from collections import Counter
import soundfile as sf
import numpy as np

BASE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
LOGS_DIR = BASE_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)


### Dataset Inspection Function

In [2]:
def inspect_dataset(data_dir: Path = DATA_RAW_DIR):
    class_mappings = {
        "Fake": ["FAKE", "fake"],
        "Real": ["Real_audio", "real", "Real", "REAL"]
    }

    results = {}
    corrupted_files = []
    all_durations = {"Fake": [], "Real": []}
    all_sample_rates = {"Fake": Counter(), "Real": Counter()}
    all_channels = {"Fake": Counter(), "Real": Counter()}
    all_extensions = {"Fake": Counter(), "Real": Counter()}

    for label, folder_names in class_mappings.items():
        class_folder = None
        for name in folder_names:
            candidate = data_dir / name
            if candidate.exists() and candidate.is_dir():
                class_folder = candidate
                break

        if not class_folder:
            print(f"Could not find folder for class '{label}' in {data_dir}")
            continue

        print(f"Inspecting class '{label}' in: {class_folder}")
        files = list(class_folder.glob("*.*"))
        total_files = len(files)
        print(f"Total files detected in '{label}': {total_files}")

        for f in files:
            all_extensions[label][f.suffix.lower()] += 1

        valid_count = 0
        for f in files:
            try:
                info = sf.info(str(f))
                valid_count += 1
                all_durations[label].append(info.duration)
                all_sample_rates[label][info.samplerate] += 1
                ch_type = "mono" if info.channels == 1 else ("stereo" if info.channels == 2 else f"{info.channels}ch")
                all_channels[label][ch_type] += 1
            except Exception as e:
                corrupted_files.append({
                    "path": str(f),
                    "filename": f.name,
                    "class": label,
                    "extension": f.suffix.lower(),
                    "error": str(e)
                })

        durs = all_durations[label]
        dur_stats = {
            "count": len(durs),
            "min_duration_sec": float(np.min(durs)) if durs else 0.0,
            "max_duration_sec": float(np.max(durs)) if durs else 0.0,
            "mean_duration_sec": float(np.mean(durs)) if durs else 0.0,
            "median_duration_sec": float(np.median(durs)) if durs else 0.0,
            "std_duration_sec": float(np.std(durs)) if durs else 0.0,
        }

        results[label] = {
            "total_files": total_files,
            "valid_audio_files": valid_count,
            "corrupted_or_unsupported": total_files - valid_count,
            "extensions": dict(all_extensions[label]),
            "sample_rates": dict(all_sample_rates[label]),
            "channel_configurations": dict(all_channels[label]),
            "duration_statistics": dur_stats
        }

    # Save reports
    unreadable_log_path = LOGS_DIR / "corrupted_or_unreadable_files.json"
    with open(unreadable_log_path, "w", encoding="utf-8") as f:
        json.dump(corrupted_files, f, indent=2)

    summary_path = LOGS_DIR / "dataset_inspection_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    return results, corrupted_files


In [3]:
# Run Inspection and Display Summary
results, corrupted_files = inspect_dataset()

print("\n" + "="*70)
print("           DATASET EXPLORATION & INSPECTION SUMMARY           ")
print("="*70)
for label, stats in results.items():
    print(f"\n[Class: {label.upper()}]")
    print(f"  - Total File Count:          {stats['total_files']:,}")
    print(f"  - Valid Readable Audio:     {stats['valid_audio_files']:,}")
    print(f"  - Corrupted / Unsupported:  {stats['corrupted_or_unsupported']:,}")
    print(f"  - File Formats:             {stats['extensions']}")
    print(f"  - Sampling Rates (Hz):      {stats['sample_rates']}")
    print(f"  - Channels (Mono/Stereo):   {stats['channel_configurations']}")
    d_stats = stats['duration_statistics']
    print(f"  - Duration: Min={d_stats['min_duration_sec']:.2f}s | "
          f"Max={d_stats['max_duration_sec']:.2f}s | "
          f"Mean={d_stats['mean_duration_sec']:.2f}s | "
          f"Median={d_stats['median_duration_sec']:.2f}s")

print(f"\nTotal Corrupted / Unreadable Audio Logged: {len(corrupted_files):,}")


Inspecting class 'Fake' in: e:\Deepfake\data\raw\FAKE
Total files detected in 'Fake': 25436
Inspecting class 'Real' in: e:\Deepfake\data\raw\Real_audio
Total files detected in 'Real': 32406

           DATASET EXPLORATION & INSPECTION SUMMARY           

[Class: FAKE]
  - Total File Count:          25,436
  - Valid Readable Audio:     25,436
  - Corrupted / Unsupported:  0
  - File Formats:             {'.wav': 56, '.flac': 25380}
  - Sampling Rates (Hz):      {44100: 52, 40000: 4, 16000: 25380}
  - Channels (Mono/Stereo):   {'stereo': 56, 'mono': 25380}
  - Duration: Min=0.65s | Max=600.00s | Mean=4.45s | Median=3.21s

[Class: REAL]
  - Total File Count:          32,406
  - Valid Readable Audio:     31,405
  - Corrupted / Unsupported:  1,001
  - File Formats:             {'.wav': 8, '.m4a': 1001, '.flac': 31397}
  - Sampling Rates (Hz):      {48000: 3, 44100: 5, 16000: 31397}
  - Channels (Mono/Stereo):   {'stereo': 8, 'mono': 31397}
  - Duration: Min=1.46s | Max=600.45s | Mean=4.18s 